In [ ]:
%load_ext autoreload
%autoreload 2

import yaml
import pandas as pd
import numpy as np
from plotnine import *

from anngeno import AnnGeno
from scripts import get_burdens, get_correlation

In [ ]:
config_path = './config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

maf = config.get("association_testing_maf")
associations_df_path = config.get("associations_df_path")
associations_df = pd.read_parquet(associations_df_path)
genes = associations_df.gene.unique()
rare_variant_annotations_dict = config.get('rare_variant_annotations') # Get the nested dict
all_annotation_list = [] # Initialize empty list
if rare_variant_annotations_dict: # Flatten the nested dict into a single list
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)
        
print("Loading AnnGeno file")
anngeno_file = config.get("anngeno_file")
ag = AnnGeno(filename=anngeno_file, filemode="r")

print(f"Filtering for variants with MAF < {maf}")
variants_to_keep = set(ag.annotations.query("MAF < @maf & region in @genes")["id"])
ag.subset_variants(variants_to_keep)

In [ ]:
gene = ag.get_region(5287)
geno = gene["genotypes"]
no_variant_mask = geno.sum(axis = 1) == 0
no_variant_mask

In [ ]:
gene["annotations"][all_annotation_list].isna().sum()

In [ ]:
config_path = './config.yaml'
zarr_burdens_path = '/s/project/deeprvat/ukb_gym/burdens/burdens_onlysum_no_vars_na.zarr'

In [ ]:
# Compute burdens
get_burdens.compute_burdens(config_path, zarr_burdens_path, max_burden=False)

In [ ]:
anno_scores_path = '/s/project/deeprvat/ukb_gym/flashzoi/flashzoi_snp_scores.parquet'

get_burdens.create_new_anno_burdens(config_path, zarr_burdens_path, anno_scores_path, max_burden=False)

In [ ]:
rho_df = get_correlation.compute_correlations(config_path, zarr_burdens_path, max_burden=False)

rho_df

In [ ]:
rank_corr_df = rho_df
# rank_corr_df = pd.read_parquet('/s/project/deeprvat/ukb_gym/data/spearman_rho/correlation_all_no_vars_na.parquet')
# rank_corr_df = rank_corr_df[~rank_corr_df['annotation'].isin(annotations_to_exclude)]
config_path = './config.yaml'  # Or wherever your config file is

with open(config_path) as f:
    config = yaml.safe_load(f)

rare_variant_annotations_dict = config.get('rare_variant_annotations')

# Create a mapping from annotation name to category
annotation_category_map = {}
if rare_variant_annotations_dict:
    for category_name, annotations in rare_variant_annotations_dict.items():
        for annotation in annotations:
            annotation_category_map[annotation] = category_name

# Add a 'category' column to rank_corr_df based on the mapping
rank_corr_df['category'] = rank_corr_df['annotation'].map(annotation_category_map)
rank_corr_df['category'] = pd.Categorical(rank_corr_df['category'], categories=['plof', 'missense', 'splicing', 'regulatory', 'misc'], ordered=True)
rank_corr_df

In [ ]:
rank_corr_df['abs_correlation'] = np.abs(rank_corr_df['spearman_correlation'])
rank_corr_df['median_abs_correlation'] = rank_corr_df.groupby('annotation')['abs_correlation'].transform('median')
rank_corr_df = rank_corr_df.sort_values('median_abs_correlation', ascending=False)
rank_corr_df['annotation'] = pd.Categorical(rank_corr_df['annotation'], categories=rank_corr_df['annotation'].unique(), ordered=True)

(
    ggplot(rank_corr_df.query("aggregation == 'sum'"), aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    # scale_y_log10() +
    # scale_y_sqrt() +
    ylab('|rank correlation|') +
    facet_wrap('~category', scales='free') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(12, 10),
    )
)

In [ ]:
(
    ggplot(rank_corr_df.query("(aggregation == 'sum') & (category == 'regulatory')"), aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    # scale_y_log10() +
    # scale_y_sqrt() +
    ylab('|rank correlation|') +
    facet_wrap('~category', scales='free') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(8, 7),
    )
)

# Phenotype vs GIS plot

In [ ]:
import zarr
from scripts import get_correlation

def pheno_gis_plot(phenotype, gene_num, annotation, burden_type, config_path, zarr_burdens_path):

    with open(config_path) as f:
        config = yaml.safe_load(f)

    anngeno_file = config.get('anngeno_file')
    # annotation_list = config.get('rare_variant_annotations')
    # phenotypes = config.get('phenotypes_for_association_testing')
    covs = config.get('covariates')
    prs_pheno_map_file = config.get('prs_pheno_map_file')
    prs_file = config.get('prs_file')

    cov_pheno_df = pd.read_parquet(f"{anngeno_file}/phenotypes.parquet", columns=['sample'] + covs + [phenotype]).set_index('sample')
    prs_df = pd.read_parquet(prs_file)
    prs_df = prs_df[prs_df.index.isin(cov_pheno_df.index)]
    prs_pheno_map = pd.read_csv(prs_pheno_map_file)
    prs_pheno_map = dict(zip(prs_pheno_map["phenotype"], prs_pheno_map["pgs_id"]))
    all_df = pd.concat([cov_pheno_df, prs_df], axis=1)
    pheno_corrected_df = get_correlation.cov_prs_correction(all_df, [phenotype], covs, prs_pheno_map)

    zarr_group = zarr.open_group(zarr_burdens_path, mode="r")
    sample_list = zarr_group['samples'][:]
    gene_list = zarr_group["genes"][:]
    annotation_list = zarr_group["annotations"][:]

    gene_idx = np.where(gene_list == gene_num)[0][0]
    anno_idx = np.where(annotation_list == annotation)[0][0]
    burden_type = burden_type.lower()
    if burden_type == "max":
        burdens = zarr_group["max_burdens"][:, gene_idx, anno_idx]
    elif burden_type == "sum":
        burdens = zarr_group["sum_burdens"][:, gene_idx, anno_idx]
    else:
        raise ValueError("burden_type must be either 'max' or 'sum'")

    burden_df = pd.DataFrame(burdens, index=sample_list, columns=[gene_num]).merge(pheno_corrected_df, left_index=True, right_on='sample')
    burden_df = burden_df.dropna()
    gis_mode = burden_df[gene_num].mode()[0]
    burden_df_non_zero = burden_df[burden_df[gene_num]!=gis_mode]
    correlation = burden_df[[gene_num, phenotype]].dropna().corr(method='spearman').iloc[0, 1]

    return burden_df, burden_df_non_zero, correlation

In [ ]:
phenotype = 'LDL_direct_statin_corrected'
gene_num = '9138'
annotation = 'alphamissense'
burden_type = 'sum'

plt_df, plt_df_nz, c = pheno_gis_plot(phenotype, gene_num, "model_22_max", burden_type, config_path, zarr_burdens_path)

(
    ggplot(plt_df, aes(x=gene_num, y=phenotype)) +
    geom_point(alpha=0.5) +
    geom_smooth(method='lm', se=False) +
    labs(x='LDLR',
         y=phenotype) +
    theme_bw() +
    labs(title = f"AbSplice2 - {round(c, 4)}")

)